In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Average
from tensorflow.keras.applications import MobileNetV2, ResNet50, InceptionV3, Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
import cv2

ModuleNotFoundError: No module named 'tensorflow'

: 

In [ ]:
activity_data_dir = r'C:\\Users\\karth\\Downloads\\activity recognition\\CSAD\\data\\activity\\train'
facial_data_dir = r'C:\\Users\\karth\\Downloads\\activity recognition\\CSAD\\data\\faces\\train'
output_dir = r'C:\\Users\\karth\\Downloads\\activity recognition\\CSAD\\output'

# Create data generators with the correct target sizes
activity_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
facial_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Activity data generator
activity_train_generator = activity_datagen.flow_from_directory(
    activity_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
activity_val_generator = activity_datagen.flow_from_directory(
    activity_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Facial data generator with correct target size
facial_train_generator = facial_datagen.flow_from_directory(
    facial_data_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
facial_val_generator = facial_datagen.flow_from_directory(
    facial_data_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 3325 images belonging to 3 classes.
Found 829 images belonging to 3 classes.
Found 35 images belonging to 3 classes.
Found 8 images belonging to 3 classes.


In [3]:
# Extract the labels dynamically from the data generators
activity_labels = list(activity_train_generator.class_indices.keys())
facial_labels = list(facial_train_generator.class_indices.keys())

# Define model creation functions
def create_model(base_model, input_shape, num_classes):
    base_model = base_model(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # Freeze the base model

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Create models
activity_models = {
    'ResNet50': create_model(ResNet50, (224, 224, 3), len(activity_labels)),
    'InceptionV3': create_model(InceptionV3, (224, 224, 3), len(activity_labels)),
    'Xception': create_model(Xception, (224, 224, 3), len(activity_labels))
}

facial_models = {
    'MobileNetV2': create_model(MobileNetV2, (160, 160, 3), len(facial_labels)),
    'InceptionV3': create_model(InceptionV3, (160, 160, 3), len(facial_labels)),
    'Xception': create_model(Xception, (160, 160, 3), len(facial_labels))
}

In [4]:
# Train and save the models
def train_and_save_models(models, train_generator, val_generator, folder_name):
    histories = {}
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    for name, model in models.items():
        print(f"Training {name}...")
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=10,
            steps_per_epoch=len(train_generator),
            validation_steps=len(val_generator)
        )
        histories[name] = history
        model.save(os.path.join(folder_name, f'{name}_model.h5'))
    return histories

# Train the activity and facial models
activity_histories = train_and_save_models(
    activity_models,
    activity_train_generator,
    activity_val_generator,
    os.path.join(output_dir, 'activity_models')
)

facial_histories = train_and_save_models(
    facial_models,
    facial_train_generator,
    facial_val_generator,
    os.path.join(output_dir, 'facial_models')
)

Training ResNet50...
Epoch 1/10
104/104 [==============================] - 73s 551ms/step - loss: 0.9797 - accuracy: 0.5317 - val_loss: 0.7692 - val_accuracy: 0.7382
Epoch 2/10
104/104 [==============================] - 45s 430ms/step - loss: 0.7177 - accuracy: 0.7158 - val_loss: 0.6145 - val_accuracy: 0.7310
Epoch 3/10
104/104 [==============================] - 48s 458ms/step - loss: 0.5729 - accuracy: 0.7907 - val_loss: 0.5467 - val_accuracy: 0.7575
Epoch 4/10
104/104 [==============================] - 47s 447ms/step - loss: 0.4727 - accuracy: 0.8328 - val_loss: 0.5008 - val_accuracy: 0.7563
Epoch 5/10
104/104 [==============================] - 48s 458ms/step - loss: 0.4140 - accuracy: 0.8490 - val_loss: 0.4835 - val_accuracy: 0.7732
Epoch 6/10
104/104 [==============================] - 46s 437ms/step - loss: 0.3905 - accuracy: 0.8556 - val_loss: 0.4608 - val_accuracy: 0.7696
Epoch 7/10
104/104 [==============================] - 47s 447ms/step - loss: 0.3507 - accuracy: 0.8680 - val_

In [5]:
# Plot and save accuracy and loss
def plot_and_save(history, model_name, folder_name):
    # Accuracy plot
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(f'{model_name} Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title(f'{model_name} Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.savefig(os.path.join(folder_name, f'{model_name}_performance.png'))
    plt.close()

def plot_all_histories(histories, folder_name):
    for name, history in histories.items():
        plot_and_save(history, name, folder_name)

# Plotting the histories
plot_all_histories(activity_histories, os.path.join(output_dir, 'activity_plots'))
plot_all_histories(facial_histories, os.path.join(output_dir, 'facial_plots'))

In [6]:
# Evaluate the models on test sets
def evaluate_models(models, test_generator):
    evaluations = {}
    for name, model in models.items():
        evaluation = model.evaluate(test_generator)
        evaluations[name] = evaluation
        print(f"{name} Model - Test Loss: {evaluation[0]}, Test Accuracy: {evaluation[1]}")
    return evaluations

# Test data generators
test_activity_datagen = ImageDataGenerator(rescale=1./255)
test_facial_datagen = ImageDataGenerator(rescale=1./255)

test_activity_generator = test_activity_datagen.flow_from_directory(
    r'C:\Users\HP\CSAD\data\activity\test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_facial_generator = test_facial_datagen.flow_from_directory(
    r'C:\Users\HP\CSAD\data\faces\test',
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical'
)

# Evaluate the activity and facial models
activity_evaluations = evaluate_models(activity_models, test_activity_generator)
facial_evaluations = evaluate_models(facial_models, test_facial_generator)

Found 3174 images belonging to 3 classes.
Found 43 images belonging to 3 classes.
100/100 [==============================] - 24s 241ms/step - loss: 0.2096 - accuracy: 0.9228
ResNet50 Model - Test Loss: 0.20962151885032654, Test Accuracy: 0.9228103160858154
100/100 [==============================] - 21s 208ms/step - loss: 0.0536 - accuracy: 0.9830
InceptionV3 Model - Test Loss: 0.0536164753139019, Test Accuracy: 0.9829867482185364
100/100 [==============================] - 21s 209ms/step - loss: 0.0612 - accuracy: 0.9814
Xception Model - Test Loss: 0.0611567422747612, Test Accuracy: 0.9814114570617676
2/2 [==============================] - 1s 708ms/step - loss: 0.2093 - accuracy: 0.8837
MobileNetV2 Model - Test Loss: 0.20928160846233368, Test Accuracy: 0.8837209343910217
2/2 [==============================] - 2s 1s/step - loss: 0.5154 - accuracy: 0.9070
InceptionV3 Model - Test Loss: 0.5154168605804443, Test Accuracy: 0.9069767594337463
2/2 [==============================] - 1s 608ms/st

In [8]:
import json

In [9]:
# Generate and save confusion matrix and classification report
def save_classification_metrics(models, test_generator, labels, folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    for name, model in models.items():
        y_true = test_generator.classes
        y_pred = model.predict(test_generator)
        y_pred_classes = np.argmax(y_pred, axis=1)

        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred_classes)
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title(f'{name} Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.savefig(os.path.join(folder_name, f'{name}_confusion_matrix.png'))
        plt.close()

        # Classification Report
        report = classification_report(y_true, y_pred_classes, target_names=labels, output_dict=True)
        report_path = os.path.join(folder_name, f'{name}_classification_report.json')
        with open(report_path, 'w') as f:
            json.dump(report, f)

# Save metrics for activity and facial models
save_classification_metrics(
    activity_models,
    test_activity_generator,
    activity_labels,
    os.path.join(output_dir, 'activity_metrics')
)

save_classification_metrics(
    facial_models,
    test_facial_generator,
    facial_labels,
    os.path.join(output_dir, 'facial_metrics')
)

2/2 [==============================] - 2s 280ms/step


In [10]:
# Real-time prediction with ensemble model
def real_time_prediction():
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Resize and preprocess frames for models
        activity_frame = cv2.resize(frame, (224, 224))
        activity_frame = np.expand_dims(activity_frame, axis=0) / 255.0

        facial_frame = cv2.resize(frame, (160, 160))
        facial_frame = np.expand_dims(facial_frame, axis=0) / 255.0

        # Ensemble Predictions for Activity
        activity_predictions = [model.predict(activity_frame) for model in activity_models.values()]
        avg_activity_prediction = np.mean(activity_predictions, axis=0)
        activity_label = activity_labels[np.argmax(avg_activity_prediction)]

        # Ensemble Predictions for Facial
        facial_predictions = [model.predict(facial_frame) for model in facial_models.values()]
        avg_facial_prediction = np.mean(facial_predictions, axis=0)
        facial_label = facial_labels[np.argmax(avg_facial_prediction)]

        # Display predictions
        cv2.putText(frame, f'Activity: {activity_label}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f'Facial Expression: {facial_label}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('Real-Time Activity & Facial Recognition', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run real-time prediction
real_time_prediction()

1/1 [==============================] - 0s 25ms/step
